# Model Comparison — Logistic Regression vs DistilBERT

Side-by-side predictions on the same emails. Shows why a transformer outperforms a bag-of-words baseline.

## 1 — Load both models

In [7]:
import pickle, sqlite3
import numpy as np
import pandas as pd
import torch
from scipy.sparse import hstack
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# ── Logistic Regression ──────────────────────────────────────────────────────
with open("model.pkl", "rb") as f:
    lr_bundle = pickle.load(f)

lr_tfidf   = lr_bundle["tfidf"]
lr_clf     = lr_bundle["clf"]
lr_numeric = lr_bundle["numeric_cols"]
lr_labels  = lr_clf.classes_

# ── DistilBERT ───────────────────────────────────────────────────────────────
BERT_DIR = "distilbert_model"
with open(f"{BERT_DIR}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_DIR)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_DIR)
bert_model.to(device)
bert_model.eval()

print("Both models loaded.")
print(f"LR classes:   {list(lr_labels)}")
print(f"BERT classes: {list(le.classes_)}")

Both models loaded.
LR classes:   ['forums', 'personal', 'promotions', 'purchases', 'social', 'spam', 'updates']
BERT classes: ['forums', 'personal', 'promotions', 'purchases', 'social', 'spam', 'updates']


## 2 — Rule-Based Layer + Prediction helpers

In [8]:
import re

# ── Rule-Based Spam Layer ────────────────────────────────────────────────────
SPAM_TLDS = {".ru", ".xyz", ".tk", ".ml", ".ga", ".cf", ".gq", ".top", ".click", ".loan"}

SPAM_KEYWORDS = {
    "winner", "won", "free", "claim", "prize", "congratulations",
    "click here", "limited time", "act now", "urgent", "verify your account",
    "you have been selected", "nigerian", "inheritance", "wire transfer",
    "100%", "guaranteed", "no risk", "make money", "earn cash",
}

def rule_based_spam_score(subject: str, body: str, sender: str = "") -> dict:
    text  = (subject + " " + body).lower()
    score = 0.0
    rules = []

    for tld in SPAM_TLDS:
        if tld in sender.lower() or tld in text:
            score += 0.4
            rules.append(f"suspicious TLD: {tld}")
            break

    hits = [kw for kw in SPAM_KEYWORDS if kw in text]
    if hits:
        score += min(0.15 * len(hits), 0.45)
        rules.append(f"keywords: {hits[:3]}")

    letters = [c for c in subject if c.isalpha()]
    if letters and sum(1 for c in letters if c.isupper()) / len(letters) > 0.4:
        score += 0.2
        rules.append("high caps in subject")

    if (subject + body).count("!") > 3:
        score += 0.15
        rules.append("excessive exclamation marks")

    return {"score": min(round(score, 2), 1.0), "rules": rules, "is_spam": score >= 0.5}


# ── Prediction helpers ───────────────────────────────────────────────────────
def count_links(text):
    return len(re.findall(r"https?://", text))

def caps_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0

def predict_lr(subject, body=""):
    text    = subject + " " + body
    X_text  = lr_tfidf.transform([text])
    numeric = np.array([[len(body), count_links(body), caps_ratio(body)]])
    X       = hstack([X_text, numeric])
    probs   = lr_clf.predict_proba(X)[0]
    return dict(zip(lr_labels, probs.round(3)))

def predict_bert(subject, body=""):
    text = subject + " " + body
    enc  = tokenizer(text, return_tensors="pt", truncation=True,
                     padding=True, max_length=128)
    enc  = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = bert_model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return dict(zip(le.classes_, probs.round(3)))

def compare(subject, body="", sender="", true_label=None):
    # 1. Rule-based check first — catches obvious spam BERT might miss
    rules = rule_based_spam_score(subject, body, sender)

    lr   = predict_lr(subject, body)
    bert = predict_bert(subject, body)

    lr_pred   = max(lr,   key=lr.get)
    bert_pred = max(bert, key=bert.get)

    # Override BERT with rules if triggered
    final_pred = "spam" if rules["is_spam"] else bert_pred
    override   = rules["is_spam"]

    rows = []
    for label in sorted(set(list(lr.keys()) + list(bert.keys()))):
        rows.append({
            "label":             label,
            "LR confidence":     lr.get(label, 0),
            "BERT confidence":   bert.get(label, 0),
        })

    df = pd.DataFrame(rows).set_index("label").sort_values("BERT confidence", ascending=False)

    print(f"Subject  : {subject}")
    if body:   print(f"Body     : {body[:80]}…")
    if sender: print(f"Sender   : {sender}")
    if true_label: print(f"True     : {true_label}")
    print(f"LR pred  : {lr_pred}  ({lr[lr_pred]:.0%})")
    print(f"BERT pred: {bert_pred}  ({bert[bert_pred]:.0%})")
    if override:
        print(f"⚠️  Rule override → SPAM  (score={rules['score']}, rules={rules['rules']})")
    print(f"✅ Final  : {final_pred.upper()}")
    print()
    return df.style.background_gradient(cmap="Blues")

## 3 — Examples from the real DB

In [9]:
# Pull one real example per category from the DB
df_db = pd.read_sql("SELECT subject, body_preview, label FROM emails", sqlite3.connect("emails.db"))
samples = df_db.groupby("label").first().reset_index()

for _, row in samples.iterrows():
    display(compare(row["subject"], row["body_preview"], true_label=row["label"]))
    print("─" * 60)

Subject  : [chris017] A security advisory on tar affects at least one of your repositories
Body     : 1 repository in your GitHub account might be affected by a security vulnerabilit…
True     : forums
LR pred  : social  (19%)
BERT pred: forums  (64%)
✅ Final  : FORUMS



,LR confidence,BERT confidence
label,,
forums,0.168000,0.639000
spam,0.131000,0.127000
updates,0.140000,0.111000
personal,0.141000,0.055000
social,0.187000,0.038000
purchases,0.135000,0.021000
promotions,0.097000,0.010000


────────────────────────────────────────────────────────────
Subject  : Christian.Schmid, knacken Sie das Sparschwein für 10€!
Body     : <div style="display:none"><img style="display:none" src="http://ae.mmstat.com/ae…
True     : personal
LR pred  : social  (19%)
BERT pred: personal  (87%)
✅ Final  : PERSONAL



,LR confidence,BERT confidence
label,,
personal,0.146000,0.868000
spam,0.132000,0.077000
promotions,0.096000,0.028000
updates,0.140000,0.010000
forums,0.159000,0.007000
social,0.191000,0.005000
purchases,0.136000,0.004000


────────────────────────────────────────────────────────────
Subject  : Also, diese Gymshark Bestellung …
Body     : 













Gymshark               96     table { border-collaps…
True     : promotions
LR pred  : forums  (24%)
BERT pred: promotions  (84%)
✅ Final  : PROMOTIONS



...

────────────────────────────────────────────────────────────
Subject  : Meet the Dune team at EthCC!
Body     : We'd love to discuss your onchain data needs

Dune (https://djFC5c04.eu1.hubsp…
True     : spam
LR pred  : social  (19%)
BERT pred: updates  (61%)
✅ Final  : UPDATES



,LR confidence,BERT confidence
label,,
updates,0.141000,0.605000
promotions,0.098000,0.224000
personal,0.140000,0.085000
spam,0.132000,0.071000
purchases,0.137000,0.012000
forums,0.160000,0.003000
social,0.192000,0.001000


────────────────────────────────────────────────────────────
Subject  : Thanks Christian, for requesting Introduction to Databricks on Azure
Body     : View in Browser <[[https://info.databricks.com/v/MDk0LVlNUy02MjkAAAGgz16iD8lMfAU…
True     : updates
LR pred  : social  (20%)
BERT pred: updates  (57%)
✅ Final  : UPDATES



,LR confidence,BERT confidence
label,,
updates,0.142000,0.566000
personal,0.152000,0.260000
promotions,0.110000,0.129000
spam,0.134000,0.028000
purchases,0.152000,0.011000
forums,0.107000,0.003000
social,0.203000,0.002000


────────────────────────────────────────────────────────────


## 4 — Custom Input

In [10]:
compare(
    subject="Congratulations! You have won a FREE iPhone 15 - Click NOW to claim",
    body="Dear winner, you have been selected. Click http://totally-legit.ru/claim to get your prize. LIMITED TIME OFFER!!!",
    sender="noreply@totally-legit.ru",
)

Subject  : Congratulations! You have won a FREE iPhone 15 - Click NOW to claim
Body     : Dear winner, you have been selected. Click http://totally-legit.ru/claim to get …
Sender   : noreply@totally-legit.ru
LR pred  : social  (19%)
BERT pred: updates  (81%)
⚠️  Rule override → SPAM  (score=1.0, rules=['suspicious TLD: .ru', "keywords: ['free', 'you have been selected', 'limited time']", 'excessive exclamation marks'])
✅ Final  : SPAM



,LR confidence,BERT confidence
label,,
updates,0.140000,0.805000
personal,0.141000,0.116000
promotions,0.099000,0.048000
purchases,0.138000,0.014000
spam,0.130000,0.012000
forums,0.159000,0.003000
social,0.194000,0.002000


In [11]:
compare(
    subject="Deine Bestellung #12345 wurde versandt",
    body="Hallo, deine Bestellung ist unterwegs. Voraussichtliche Lieferung: Montag.",
)

Subject  : Deine Bestellung #12345 wurde versandt
Body     : Hallo, deine Bestellung ist unterwegs. Voraussichtliche Lieferung: Montag.…
LR pred  : forums  (23%)
BERT pred: purchases  (49%)
✅ Final  : PURCHASES



,LR confidence,BERT confidence
label,,
purchases,0.124000,0.486000
updates,0.134000,0.325000
personal,0.128000,0.159000
spam,0.121000,0.010000
forums,0.231000,0.009000
social,0.177000,0.007000
promotions,0.085000,0.006000
